# Day 2 — Runtime Architecture

**What you will learn:**
- The three components of a Spark cluster
- How a query becomes Jobs → Stages → Tasks
- Narrow vs wide transformations — why it matters
- Deployment modes: local, client, cluster
- How Spark recovers from failures

**Exam relevance:** Architecture (20%) — execution hierarchy and deployment modes are tested directly.

## 1. Cluster Components

```
┌─────────────────────────────────────────────────┐
│                  SPARK CLUSTER                  │
│                                                 │
│  ┌──────────────┐       ┌───────────────────┐  │
│  │    DRIVER    │◄─────►│  CLUSTER MANAGER  │  │
│  │              │       │  (YARN/K8s/DBR)   │  │
│  │ SparkContext │       └───────────────────┘  │
│  │ Scheduler    │               │               │
│  │ DAG Builder  │       ┌───────┴───────┐       │
│  └──────────────┘       ▼               ▼       │
│                   ┌──────────┐  ┌──────────┐   │
│                   │EXECUTOR 1│  │EXECUTOR 2│   │
│                   │ Task Task│  │ Task Task│   │
│                   │ [cache]  │  │ [cache]  │   │
│                   └──────────┘  └──────────┘   │
└─────────────────────────────────────────────────┘
```

| Component | Role |
|---|---|
| **Driver** | Your notebook / application. Builds the execution plan, schedules tasks, collects results. |
| **Cluster Manager** | Allocates resources (workers). Options: Databricks, YARN, Kubernetes, Mesos, Standalone. |
| **Executors** | JVM processes on worker nodes. Run tasks, store cached data. Created at app start, live for its duration. |

In [ ]:
sc = spark.sparkContext

print(f"Master            : {sc.master}")
print(f"Default parallelism: {sc.defaultParallelism}")  # total cores across all executors
print(f"App name          : {sc.appName}")

## 2. Jobs → Stages → Tasks

When you call an **action** (`.show()`, `.count()`, `.write()`), Spark creates a **Job**.  
Each job is broken into **Stages** at shuffle boundaries.  
Each stage is split into **Tasks** — one task per partition.

```
ACTION called
    └── JOB
         ├── STAGE 1  (narrow transformations — no shuffle)
         │     ├── Task (partition 0)
         │     ├── Task (partition 1)
         │     └── Task (partition N)
         └── STAGE 2  (after shuffle boundary)
               ├── Task (partition 0)
               └── Task (partition 1)
```

> **Rule:** Every shuffle creates a new stage boundary.

In [ ]:
data = [(i, f"user_{i % 5}", i * 10) for i in range(1, 21)]
df = spark.createDataFrame(data, ["id", "user", "amount"])

# Check how many partitions this DataFrame has
print(f"Partitions: {df.rdd.getNumPartitions()}")

# This groupBy triggers a shuffle → 2 stages
result = df.groupBy("user").sum("amount")
result.show()
# Open the Spark UI (cluster → Spark UI tab) to see the 2 stages

## 3. Narrow vs Wide Transformations

This is a core exam concept — know the difference cold.

| | Narrow | Wide |
|---|---|---|
| **Data movement** | None — each partition is processed independently | Shuffle — data moves across the network |
| **Stage boundary** | No | Yes — creates a new stage |
| **Examples** | `filter`, `select`, `withColumn`, `map`, `union` | `groupBy`, `join`, `distinct`, `repartition`, `orderBy` |
| **Performance** | Fast | Slower (network I/O) |

```
NARROW (filter)              WIDE (groupBy)

P0 ──filter──► P0'          P0 ─┐
P1 ──filter──► P1'          P1 ─┼──shuffle──► P0' (grouped)
P2 ──filter──► P2'          P2 ─┘             P1' (grouped)

No data crosses partitions   Data redistributed across network
```

In [ ]:
from pyspark.sql.functions import col

# Narrow — filter stays within each partition, no shuffle
narrow = df.filter(col("amount") > 50).select("id", "amount")
narrow.explain()  # 1 stage expected

# Wide — groupBy requires data from all partitions to be redistributed
wide = df.groupBy("user").count()
wide.explain()  # 2 stages: scan+shuffle write, then shuffle read+aggregate

## 4. Deployment Modes

| Mode | Driver location | Typical use |
|---|---|---|
| **local** | Same machine as executor | Local dev/testing — `local[*]` uses all cores |
| **client** | Machine submitting the job | Interactive notebooks (Databricks, Jupyter) |
| **cluster** | A worker node in the cluster | Production batch jobs (`spark-submit`) |

> **Exam tip:** In **client mode**, if the submitting machine fails, the Driver dies and the job fails.  
> In **cluster mode**, the Driver runs inside the cluster — the submitting machine can disconnect safely.

In Databricks, you always run in **client mode** from the notebook.

In [ ]:
# Verify current mode and config
print(spark.conf.get("spark.submit.deployMode", "client (default)"))
print(spark.conf.get("spark.executor.memory", "not set"))
print(spark.conf.get("spark.executor.cores", "not set"))

## 5. Fault Tolerance via Lineage

Spark does **not** replicate data between steps. Instead it remembers the **lineage** — the full sequence of transformations that produced each partition.

If an executor dies mid-job:
1. The Cluster Manager detects the failure
2. Spark re-schedules only the **failed tasks** on a healthy executor
3. The lineage is replayed from the last checkpoint or from the source

> **Exam tip:** Spark's fault tolerance model is **lineage-based recomputation**, not replication. RDDs store the recipe, not copies of the data.

In [ ]:
# toDebugString shows the lineage (dependency chain) of an RDD
rdd = df.filter(col("amount") > 50).select("user").rdd
print(rdd.toDebugString().decode())

---

## Day 2 Checklist

- [ ] Can draw the Driver / Cluster Manager / Executor triangle from memory
- [ ] Understand: Action → Job → Stages → Tasks
- [ ] Know which transformations are narrow and which are wide
- [ ] Can explain why groupBy creates a stage boundary (shuffle)
- [ ] Know the 3 deployment modes and when each is used
- [ ] Understand fault tolerance via lineage recomputation

**Next:** Day 3 — Lazy Evaluation & the Catalyst Optimizer